# 01 · Parse & inspect
Step 1 of the build plan: extract text and tables from all 9 filings and sanity-check the table extraction before any chunking.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import Markdown, display
from finsight.config import list_documents, FISCAL_YEAR_END
from finsight.ingest.parsers import parse_document
docs = list_documents()
pd.DataFrame([{'doc_id': d.doc_id, 'company': d.company, 'fiscal_year': d.fiscal_year, 'type': d.doc_type, 'file': d.path.name} for d in docs])

## Parse every document

In [ ]:
parsed = {d.doc_id: parse_document(d) for d in docs}
rows = []
for d in docs:
    els = parsed[d.doc_id]
    tb = [e for e in els if e.kind == 'table']
    rows.append({'doc_id': d.doc_id, 'text_blocks': sum(e.kind == 'text' for e in els), 'text_chars': sum(len(e.text) for e in els if e.kind == 'text'),
                 'tables': len(tb), 'median_rows': pd.Series([len(t.rows) for t in tb]).median(), 'max_rows': max(len(t.rows) for t in tb)})
summary = pd.DataFrame(rows).set_index('doc_id'); summary

## Eyeball tables
Pick a document and a table index. Check that headers, units and numbers survived.

In [ ]:
def show(doc_id, i):
    tb = [e for e in parsed[doc_id] if e.kind == 'table']
    t = tb[i]
    print(f'{doc_id} · table {i}/{len(tb)} · page {t.page} · {len(t.rows)} rows')
    display(Markdown(t.text[:3000]))
show('amazon_2024', 5)

In [ ]:
show('alphabet_2024', 10)

In [ ]:
show('microsoft_2024', 10)

## Find a known table
Search for a table containing a label we expect, to confirm the key financial statements were captured as tables.

In [ ]:
def find(doc_id, needle):
    return [(i, t.page, len(t.rows)) for i, t in enumerate(e for e in parsed[doc_id] if e.kind == 'table') if needle.lower() in t.text.lower()]
for doc, needle in [('amazon_2024', 'Total net sales'), ('alphabet_2024', 'Total revenues'), ('microsoft_2024', 'Intelligent Cloud')]:
    print(doc, needle, find(doc, needle)[:8])

## Verify metadata
Confirm the fiscal year inferred from each file name by looking at the opening text.

In [ ]:
for d in docs:
    first = ' '.join(e.text for e in parsed[d.doc_id][:3] if e.kind == 'text')[:200].replace(chr(10), ' ')
    print(d.doc_id, '|', FISCAL_YEAR_END[d.company], '|', first)

## Notes / issues found
_(fill in after reviewing the samples above)_